# DynamoDB Concepts

## Overview
**Amazon DynamoDB** is a fully managed, serverless, key-value and document database. It provides single-digit millisecond performance at any scale, with no servers to manage.

**DynamoDB vs MongoDB vs PostgreSQL:**

| | DynamoDB | MongoDB | PostgreSQL |
|---|---|---|---|
| Model | Key-value + document | Document | Relational |
| Query flexibility | Low (key-based only) | Medium (any field) | High (arbitrary SQL) |
| Scale | Horizontal, auto | Horizontal (sharding) | Vertical primary |
| Ops burden | Zero (fully managed) | Low-medium | Medium |
| Cost model | Per request / provisioned | Instance-based | Instance-based |
| JOINs | No | $lookup (pipeline) | Yes (full SQL) |
| ACID transactions | Single-item atomic; multi-item supported | Multi-doc (4.0+) | Full ACID |
| Best for | High-scale key-based access | Flexible documents | Complex analytics/relations |

**When DynamoDB is the right choice:**
- Known, predictable access patterns that can be served by key lookups
- Need for infinite horizontal scale without operational overhead
- Serverless architectures (Lambda + API Gateway + DynamoDB)
- Session stores, leaderboards, IoT sensor data, user profiles
- Healthcare/finance apps where AWS infrastructure is already in use

**Local development:** Use DynamoDB Local (Docker: `amazon/dynamodb-local`) to develop without AWS credentials. All boto3 code in this folder uses `endpoint_url='http://localhost:8000'`.

---

In [1]:

print("=== DynamoDB core terminology ===")
terms = [
    ("Table",            "SQL: Table",          "Top-level container; each item can have different attributes"),
    ("Item",             "SQL: Row",            "A collection of attributes; max 400 KB"),
    ("Attribute",        "SQL: Column",         "Key-value pair; type is per-item, not per-table"),
    ("Partition Key (PK)","SQL: Primary Key part 1", "Hash of PK determines which partition stores the item"),
    ("Sort Key (SK)",    "SQL: Primary Key part 2", "Optional; enables range queries within a partition"),
    ("Primary Key",      "SQL: Composite PK",   "PK alone (simple) or PK+SK (composite) — must be unique"),
    ("GSI",              "SQL: Index on any columns", "Global Secondary Index; different PK+SK; separate capacity"),
    ("LSI",              "SQL: Index with same PK",   "Local Secondary Index; same PK, different SK; must be created at table creation"),
    ("Capacity Unit",    "SQL: N/A",            "RCU (Read) and WCU (Write) are the billing/throughput units"),
    ("On-Demand mode",   "SQL: N/A",            "Pay per request; no capacity planning; auto-scales instantly"),
    ("Provisioned mode", "SQL: N/A",            "Set RCU/WCU; cheaper at predictable workloads; supports auto-scaling"),
    ("Stream",           "SQL: Change Data Capture", "Ordered log of item changes; triggers Lambda; 24-hour retention"),
    ("TTL",              "SQL: N/A",            "Auto-expire items by setting an epoch timestamp attribute"),
    ("Condition Expression","SQL: CHECK constraint + WHERE in writes","Prevents writes unless a condition is met"),
    ("Projection Expression","SQL: SELECT column list","Limits which attributes are returned by reads"),
]
print(f"  {'DynamoDB':<22s}  {'SQL analogue':<28s}  Description")
print("  " + "-"*85)
for term, sql, desc in terms:
    print(f"  {term:<22s}  {sql:<28s}  {desc}")


=== DynamoDB core terminology ===
  DynamoDB                SQL analogue                  Description
  -------------------------------------------------------------------------------------
  Table                   SQL: Table                    Top-level container; each item can have different attributes
  Item                    SQL: Row                      A collection of attributes; max 400 KB
  Attribute               SQL: Column                   Key-value pair; type is per-item, not per-table
  Partition Key (PK)      SQL: Primary Key part 1       Hash of PK determines which partition stores the item
  Sort Key (SK)           SQL: Primary Key part 2       Optional; enables range queries within a partition
  Primary Key             SQL: Composite PK             PK alone (simple) or PK+SK (composite) — must be unique
  GSI                     SQL: Index on any columns     Global Secondary Index; different PK+SK; separate capacity
  LSI                     SQL: Index with same PK 

---
## Primary key design

In [3]:
print("=== Primary key design: the most important DynamoDB decision ===")
key_examples = [
    ("Simple PK only",
     "site_id (PK)",
     "Fast single-site lookups. No range queries within a site.",
     "sites table: get all attributes for site_id = 'SITE#001'"),
    ("Composite PK + SK",
     "site_id (PK)  |  obs_date#obs_id (SK)",
     "Query all observations for a site; range on date; get one observation.",
     "observations table: Query site_id='SITE#001', SK begins_with '2023'"),
    ("Single-table composite",
     "PK            |  SK",
     "Multiple entity types in one table. PK+SK encode entity type.",
     """PK='SITE#001', SK='SITE#001'              → site record
     PK='SITE#001', SK='OBS#2023-04-10#001'    → observation
     PK='SITE#001', SK='WQ#2023-04-10'         → water quality"""),
]
print("Key design patterns:")
for name, key_structure, notes, example in key_examples:
    print(f"  Pattern: {name}")
    print(f"  Keys:    {key_structure}")
    print(f"  Notes:   {notes}")
    print(f"  Example: {example}")
    print()

print("Cardinal rule: design your keys around your access patterns.")
print("In SQL, you query then add indexes. In DynamoDB, you design keys first.")

=== Primary key design: the most important DynamoDB decision ===
Key design patterns:
  Pattern: Simple PK only
  Keys:    site_id (PK)
  Notes:   Fast single-site lookups. No range queries within a site.
  Example: sites table: get all attributes for site_id = 'SITE#001'

  Pattern: Composite PK + SK
  Keys:    site_id (PK)  |  obs_date#obs_id (SK)
  Notes:   Query all observations for a site; range on date; get one observation.
  Example: observations table: Query site_id='SITE#001', SK begins_with '2023'

  Pattern: Single-table composite
  Keys:    PK            |  SK
  Notes:   Multiple entity types in one table. PK+SK encode entity type.
  Example: PK='SITE#001', SK='SITE#001'              → site record
     PK='SITE#001', SK='OBS#2023-04-10#001'    → observation
     PK='SITE#001', SK='WQ#2023-04-10'         → water quality

Cardinal rule: design your keys around your access patterns.
In SQL, you query then add indexes. In DynamoDB, you design keys first.


---
## Capacity modes and data types

In [4]:

print("=== Capacity modes and data types ===")
capacity = '''
# ── On-Demand mode ───────────────────────────────────────────────
# Best for: unpredictable traffic; new applications; dev/test
# Cost: ~2.5x provisioned at steady load; no capacity planning

# ── Provisioned mode ────────────────────────────────────────────
# Best for: predictable workloads; cost-sensitive production apps
# Costs: set RCU and WCU; can use auto-scaling

# ── Read Capacity Unit (RCU) definition ──────────────────────────
# 1 RCU = 1 strongly consistent read  of 1 item up to 4 KB
# 1 RCU = 2 eventually consistent reads of 1 item up to 4 KB
# Transactional reads cost 2x (2 RCU per 4 KB)

# ── Write Capacity Unit (WCU) definition ─────────────────────────
# 1 WCU = 1 write of 1 item up to 1 KB
# Transactional writes cost 2x (2 WCU per 1 KB)
'''
print("Capacity modes:")
print(capacity)

print("DynamoDB attribute types:")
types = [
    ("S",    "String",          "'SITE#001', '2023-04-10', 'Atlantic'"),
    ("N",    "Number",          "42, 7.25, -3.14  (stored as string internally)"),
    ("B",    "Binary",          "Base64-encoded bytes (images, compressed data)"),
    ("BOOL", "Boolean",         "True, False"),
    ("NULL", "Null",            "None in Python"),
    ("L",    "List",            "['hypertension', 'type2_diabetes']  (any types mixed)"),
    ("M",    "Map",             "{'email': 'a@b.com', 'phone': '506-555-0101'}"),
    ("SS",   "String Set",      "{'NB', 'ON', 'BC'}  (unique strings; no ordering)"),
    ("NS",   "Number Set",      "{1, 2, 3}  (unique numbers)"),
    ("BS",   "Binary Set",      "Set of binary values"),
]
print(f"  {'Type':<6s}  {'Name':<12s}  Example")
print("  " + "-"*60)
for code, name, example in types:
    print(f"  {code:<6s}  {name:<12s}  {example}")


=== Capacity modes and data types ===
Capacity modes:

# ── On-Demand mode ───────────────────────────────────────────────
# Best for: unpredictable traffic; new applications; dev/test
# Cost: ~2.5x provisioned at steady load; no capacity planning

# ── Provisioned mode ────────────────────────────────────────────
# Best for: predictable workloads; cost-sensitive production apps
# Costs: set RCU and WCU; can use auto-scaling

# ── Read Capacity Unit (RCU) definition ──────────────────────────
# 1 RCU = 1 strongly consistent read  of 1 item up to 4 KB
# 1 RCU = 2 eventually consistent reads of 1 item up to 4 KB
# Transactional reads cost 2x (2 RCU per 4 KB)

# ── Write Capacity Unit (WCU) definition ─────────────────────────
# 1 WCU = 1 write of 1 item up to 1 KB
# Transactional writes cost 2x (2 WCU per 1 KB)

DynamoDB attribute types:
  Type    Name          Example
  ------------------------------------------------------------
  S       String        'SITE#001', '2023-04-10', 'Atlant

---
## Common Pitfalls

**1. Designing keys after building the application**
In SQL you can add an index anytime. In DynamoDB, the partition key and sort key are fixed at table creation. A table with the wrong key structure must be entirely rebuilt and migrated. Always map out every access pattern (list all sites, get one site, get observations for site by date range) before designing the key schema.

**2. Using a low-cardinality partition key**
If all items share a small number of partition key values (e.g. `status = 'active'|'inactive'`), all traffic hits one or two partitions -- a hot partition. DynamoDB distributes data by partition key hash; you need high cardinality (thousands of distinct values) to spread load evenly. A `site_id` with 10 sites is borderline; a `user_id` with millions of users is ideal.

**3. Confusing On-Demand and Provisioned billing**
On-Demand charges per request at ~2.5x the cost of Provisioned at steady throughput. A production table with 10,000 reads/second on On-Demand can cost 2.5x what Provisioned would cost. Start with On-Demand for new tables; switch to Provisioned with auto-scaling once traffic patterns stabilise.

**4. Storing items larger than 400 KB**
DynamoDB enforces a hard 400 KB limit per item. Storing large blobs (images, PDFs, lengthy notes) inside an item will fail. Store binary data in S3 and keep only the S3 key in DynamoDB.

**5. Not using TTL for time-limited data**
Session tokens, temporary locks, and rate-limit counters that are no longer valid should be automatically expired. Not setting a TTL attribute means stale items accumulate, increasing storage costs and polluting scan results. TTL deletion is free and eventual (within 48 hours).

**6. Treating DynamoDB like a relational database**
DynamoDB has no JOINs, no GROUP BY, no aggregate functions, and no arbitrary WHERE clauses. Every query must go through the primary key or an index. If your access patterns require ad-hoc queries across many attributes, consider using DynamoDB Streams to sync to a PostgreSQL read replica, or use Amazon Athena over DynamoDB exports.


---
*sql_methods_library - Samantha McGarrigle*